In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
from zipfile import ZipFile

with ZipFile('archive.zip', 'r') as zipObj:
   # Extract all the contents of zip file in current directory
   zipObj.extractall()
   print('Done')

Done


In [3]:
EPOCHS = 1000
SEED = 42
learning_rate = 0.01

In [ ]:
data = pd.read_csv('/content/student_performance_data.csv')
data

In [5]:
data.describe()

,StudentID,Age,StudyHoursPerWeek,AttendanceRate,GPA
count,500.000000,500.000000,500.000000,500.000000,500.00000
mean,250.500000,20.956000,19.876000,74.990380,2.98516
std,144.481833,2.000517,11.471347,14.565917,0.56362
min,1.000000,18.000000,1.000000,50.010000,2.00000
25%,125.750000,19.000000,10.000000,62.607500,2.48750
50%,250.500000,21.000000,20.500000,75.730000,3.00000
75%,375.250000,23.000000,30.000000,87.220000,3.48000
max,500.000000,24.000000,39.000000,99.970000,3.99000


In [6]:
X = data[["StudyHoursPerWeek", "AttendanceRate", "Age"]].values
y = data["GPA"].values.reshape(-1, 1)

In [7]:
scaler = MinMaxScaler()

X = scaler.fit_transform(X)
y = scaler.fit_transform(y)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [9]:
class Performance_Prediction(tf.keras.Model):
  def __init__(self):
    super(Performance_Prediction, self).__init__()
    self.dense1 = layers.Dense(64, activation='relu')
    self.dense2 = layers.Dense(64, activation='relu')
    self.dense3 = layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

In [10]:
class PrintDot(keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs):
        if epoch % 100 == 0: print('')
        print('.', end='')
        if epoch == EPOCHS - 1 or epoch == logs.get('val_loss', None):
            print('')

In [11]:
def plot_metrics(history):
  hist = pd.DataFrame(history.history)
  hist['epoch'] = history.epoch

  print(hist.tail())

In [12]:
def plot_errors(history):
  hist = pd.DataFrame(history.history)
  hist['epoch'] = history.epoch

  plt.figure()
  plt.xlabel('Epoch')
  plt.ylabel('Mean Abs Error [GPA]')
  plt.plot(hist['epoch'], hist['mae'], label='Train Error')
  plt.plot(hist['epoch'], hist['val_mae'], label='Val Error')
  plt.ylim([0, hist['val_mae'].max()*1.1])
  plt.legend()

  plt.figure()
  plt.xlabel('Epoch')
  plt.ylabel('Mean Square Error [$GPA^2$]')
  plt.plot(hist['epoch'], hist['mse'], label='Train Error')
  plt.plot(hist['epoch'], hist['val_mse'], label='Val Error')
  plt.ylim([0, hist['val_mse'].max()*1.1])
  plt.legend()
  plt.show()

In [13]:
model = Performance_Prediction()
model.summary()

Model: "performance__prediction"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(loss='mse',
              optimizer=tf.keras.optimizers.SGD(learning_rate),
              metrics=['mae', 'mse'])

In [15]:
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10)

history = model.fit(X_train, y_train, epochs=EPOCHS,
                    validation_data = (X_test, y_test), verbose=0, callbacks=[early_stop, PrintDot()])


.....................................

In [16]:
plot_metrics(history)

        loss       mae       mse  val_loss   val_mae   val_mse  epoch
32  0.075918  0.237565  0.075918  0.091475  0.265052  0.091475     32
33  0.076002  0.237622  0.076002  0.091492  0.265108  0.091492     33
34  0.075928  0.237541  0.075928  0.091503  0.265152  0.091503     34
35  0.075879  0.237665  0.075879  0.091511  0.265216  0.091511     35
36  0.075981  0.237517  0.075981  0.091551  0.265304  0.091551     36


In [17]:
# 51  0.076265  0.236531  0.076265  0.088919  0.259472  0.088919     51
# 36  0.075981  0.237517  0.075981  0.091551  0.265304  0.091551     36

In [ ]:
plot_errors(history)